# DeepSeek 模型微调

## 1. 课程简介

本课件将带领大家学习如何使用 LoRA（Low-Rank Adaptation）技术对 Deepseek-R1-1.5B 大语言模型进行高效微调。  
LoRA 是一种参数高效的微调方法，能够在不大幅增加显存和计算资源消耗的情况下，让大模型快速适应新任务。  
通过本实训，你将掌握大模型微调的基本流程、数据处理方法、训练技巧以及效果评估方法。

**课程结构**：

1. **环境准备** - 导入必要的库和设置环境,
2. **数据准备** - 加载和处理训练数据,
3. **模型加载** - 加载预训练模型和分词器,
4. **微调前测试** - 观察原始模型的输出效果,
5. **LoRA配置** - 设置 LoRA 参数和训练配置,
6. **模型训练** - 执行微调训练过程,
7. **效果对比** - 分析微调前后的差异,

## 2. 环境准备与库导入  
在本节中，我们首先导入本次实验所需的所有`Python`库，包括`PyTorch`、`transformers`、`peft`、`datasets`等。这些库分别用于深度学习、模型加载、参数高效微调和数据处理。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import matplotlib.pyplot as plt
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'

- **torch**: PyTorch深度学习框架，提供张量运算和自动微分功能
- **transformers**: Hugging Face的 LLM 库，提供 LLM 相关的预训练模型和工具
  - `AutoTokenizer`: 自动加载适合模型的分词器
  - `AutoModelForCausalLM`: 自动加载因果语言模型
  - `TrainingArguments`: 训练参数配置类
  - `Trainer`: 训练器类，简化训练流程
  - `TrainerCallback`: 训练回调类，用于监控训练过程
- **peft**: 参数高效微调库
  - `LoraConfig`: LoRA配置类
  - `get_peft_model`: 将模型转换为PEFT模型
- **datasets**: 数据集处理库
- **matplotlib.pyplot**: 用于数据可视化和绘图
- **os**: 操作系统接口，用于文件路径操作

## 3. 数据集准备  

- model_path：指定预训练大模型的本地路径。  
- data_path：指定微调所用的中文医学问答数据集路径。  
- output_path：指定模型微调后权重的保存路径。  

In [ ]:
model_path = r"/home/jovyan/work/datasets/685268a0767d61d67a4b26f3-momodel/deepseek_finetune/deepseek_r1_1b/"  # 模型路径  
data_path = r"/home/jovyan/work/datasets/685268a0767d61d67a4b26f3-momodel/deepseek_finetune/medical_o1_sft_Chinese.json"  # 数据集路径
output_path = r"/home/jovyan/work/results"

加载和查看数据  
使用`datasets`库加载`json`格式的数据集，并可通过索引查看单条样本内容。  
本实训使用的数据集每条数据包含 Question（问题）、Complex_CoT（详细推理过程）、Response（最终答案）。

In [ ]:
# 使用 load_dataset 函数加载json格式的数据集
# data_files: 指定数据文件路径，这里使用之前定义的data_path
# split: 指定要加载的数据集部分，'train[:100]'表示加载训练集的前100条样本
dataset = load_dataset("json", data_files=data_path, split='train[:100]')  

In [ ]:
#  加载完成的数据集可以使用索引查看单条样本内容
dataset[0]

## 4. 分词器加载  
加载与模型匹配的分词器，并将样本格式化为模型输入字符串，便于后续分词和训练。

In [ ]:
# 使用 AutoTokenizer 从预训练模型路径加载分词器
# model_path: 预训练模型的本地路径
tokenizer = AutoTokenizer.from_pretrained(model_path)

### 4.1 格式化输入
由于 Deepseek-R1-1.5B 是对话模型，因此需要将每个训练样本格式化为对话的形式。  
例如我们使用训练集第一条样本的 Question 作为提示词，查看格式化后的模型输入是怎样的。  

In [ ]:
prompt = "根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？" 
messages = [
    [{"role": "system", "content": "你是一个有用的助手。"},  # 这个表示模型扮演的角色
    {"role": "user", "content": prompt}],  # 这个表示用户输入的内容
]

In [ ]:
# 使用tokenizer的apply_chat_template方法将对话格式化为模型输入
# 参数说明：
#   messages: 对话消息列表，包含系统角色和用户输入内容
#   tokenize: 是否进行分词，False表示返回格式化后的字符串
#   add_generation_prompt: 是否添加生成提示，True表示在输入末尾添加生成提示符
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True 
)

In [ ]:
text

使用 deepseek 的 `tokenizer.apply_chat_template` 方法格式化后，得到的序列以特殊标记开头和分隔不同角色的内容。

1. `<｜begin▁of▁sentence｜>`：表示序列的起始，帮助模型识别输入的开始。
2. `<｜User｜>` 和 `<｜Assistant｜>`：分别标记用户和助手的发言，明确对话的角色分界，便于模型理解上下文。
3. `<think>\n`：是助手回答的起始提示，通常用于引导模型输出思考过程或答案。
4. 这种结构化的模板有助于**对话模型**更好地学习和生成符合预期的多轮对话内容，保证输入输出的一致性和可控性。

### 4.2 分词

现在我们已经得到了格式化的文本，这些内容对于人类来说很容易理解，但计算机并不能直接“读懂”这些汉字或英文单词的含义。  
那么，计算机是如何理解和处理这些自然语言的呢？这就需要用到“分词”和“token（标记）”的技术。  
分词，简单来说，就是把一段连续的文本按照一定的规则切分成一个个有意义的“词”或“子词”单元。例如，“我爱学习”可以被分成“我”、“爱”、“学习”三个词。  
但在深度学习和大模型中，分词往往更细致，常常会把词进一步拆分成更小的“子词”或“字符”，这样可以更好地处理各种新词、专有名词和不同语言的混合文本。  
分词后的每个“词”或“子词”都会被赋予一个唯一的编号，这个编号就叫做“token”。这些token编号是计算机可以直接处理的数字，模型就是通过这些数字来“理解”和“学习”语言的。  
举个例子，如果我们有一句话“你好，世界！”，分词器可能会把它分成["你", "好", "，", "世界", "！"]，然后分别对应成[101, 102, 103, 104, 105]这样的token编号。  
通过分词和token化，原本人类可读的文本就被转换成了计算机可以理解和处理的数字序列，这样模型才能进行后续的学习和推理。  

In [ ]:
# 使用分词器（tokenizer）将格式化后的文本转换为模型可接受的输入格式（张量）
# 参数说明：
#   text：待分词的输入文本（已按对话模板格式化）
#   return_tensors="pt"：返回PyTorch张量格式，便于后续送入模型
#   padding=True：自动对输入进行填充，使得批量输入时长度一致
text_inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

In [ ]:
text_inputs

input_ids 是输入文本中每个词元（token）的数值标识符。  
根据分词器的特定词汇表和规则，这些词元可以是单词、单词的一部分（子词）甚至是单个字符。分词器词汇表中的每个独特词元都被分配一个唯一的整数ID。  
可以看到 `<｜begin▁of▁sentence｜>` 对应的标识符为 151646。  
`attention_mask` 是一个二进制张量（意味着它只包含 0 和 1），它告诉模型应该关注`input_ids`的哪些部分，以及应该忽略哪些部分。  
1 表示实际输入文本对应的 token, 0 表示对应位置为无意义的填充 token。  

### 4.3 向量化（Embedding）  
在上一节中，我们已经了解了分词和token的概念。每个token其实就是一个数字编号，但仅仅有编号还不够。  
计算机并不能直接通过这些编号“理解”语言的含义。那么，模型是如何让这些数字变得有意义，从而理解语义的呢？  
这就需要用到“向量化”或“嵌入（Embedding）”技术。  

**什么是Embedding？**  
Embedding就是把每个token编号映射成一个高维的向量（比如长度为768或1024的浮点数数组）。  
这些向量不是随便生成的，而是在模型训练过程中，通过大量语料学习出来的。  
这样，语义相近的词（比如“猫”和“狗”）它们的向量距离也会很近，而语义差异大的词（比如“苹果”和“汽车”）向量距离就会远。 

**为什么Embedding能让模型理解语义？**  
1. 向量空间可以表达复杂的语义关系，比如“国王-男人+女人≈王后”。
2. 模型通过对这些向量的计算，可以捕捉到词与词之间的关联、上下文信息等。
3. 这样，原本只是编号的token，经过Embedding后，就变成了模型可以“理解”和“处理”的语义信息。  

**举个例子：**
假设“猫”被分配了token编号105，“狗”是106。  
经过Embedding层后，“猫”可能被映射为[0.12, -0.34, ..., 0.56]这样的向量，“狗”则是[0.13, -0.30, ..., 0.60]。  
这两个向量在高维空间中很接近，说明它们语义相似。



Deepseek 的 embedding 层包含在读取的模型权重中，微调的时候不需要去调整这个层。  

## 5. 构建训练数据集

现在我们将第 3 小节读取的数据集处理成可以用于微调的格式。  
首先定义数据批量处理函数`dataset_process`，将每条样本处理为模型训练所需的格式，包括：  
- 拼接指令和目标文本  
- 分词并生成input_ids、attention_mask
- 构建labels，指令部分设为-100（不参与loss计算），只让模型学习生成目标部分

In [ ]:
def dataset_process(example):
    """
    对单条样本进行格式化、分词和token处理，生成可用于大模型微调的数据格式。

    参数说明：
        example (dict): 包含'Question', 'Complex_CoT', 'Response'等字段的单条样本。

    返回：
        dict: 包含input_ids, attention_mask, labels的字典，均为张量格式。
    """

    # 1. 构建指令部分（prompt），模拟对话开头，模型只需学习回答部分
    instruction = f"你是一个有用的助手。<｜User｜>{example['Question']}<｜Assistant｜><think>\n"

    # 2. 构建目标部分（target），即模型需要生成的内容（思考链+答案）
    target = f"{example['Complex_CoT']}</think>\n\n答案：{example['Response']}<｜end▁of▁sentence｜>"

    # 3. 拼接完整输入文本（instruction + target）
    full_text = instruction + target

    # 4. 分词处理，将文本转为模型可接受的张量格式
    # data_tokenizer参数说明：
    #   - text: 输入文本
    #   - padding="max_length": 填充到最大长度
    #   - truncation=True: 超长截断
    #   - max_length=1024: 最大长度1024
    #   - return_tensors="pt": 返回PyTorch张量
    #   - add_special_tokens=True: 添加特殊token（如BOS/EOS等）
    inputs = tokenizer(
        full_text,
        padding="max_length",
        truncation=True,
        max_length=1024,
        return_tensors="pt",
        add_special_tokens=True
    )

    # 5. 计算指令部分的token长度（用于后续mask掉loss）
    # add_special_tokens=False：不添加特殊token，确保长度准确
    instruction_encoded = tokenizer(
        instruction,
        add_special_tokens=False
    )["input_ids"]
    instruction_length = len(instruction_encoded)

    # 6. 创建labels，只有target部分参与loss计算
    #   - 先复制input_ids
    #   - 将指令部分的token位置设为-100（PyTorch的CrossEntropyLoss会忽略-100）
    labels = inputs["input_ids"].clone().squeeze(0)
    labels[:instruction_length] = -100

    # 7. 返回处理后的字典，包含input_ids、attention_mask、labels
    return {
        "input_ids": inputs["input_ids"].squeeze(0),         # 输入token id
        "attention_mask": inputs["attention_mask"].squeeze(0), # 注意力mask
        "labels": labels                                      # 训练标签
    }


使用map方法批量处理整个数据集，得到可直接用于训练的格式。

In [ ]:
# 使用 map 方法对原始数据集进行处理，生成适用于模型训练的新数据集
# 参数说明：
#   - dataset: 原始数据集，需包含 'Question'、'Complex_CoT'、'Response' 字段
#   - dataset_process: 处理函数，将每条数据转为模型输入格式（含 input_ids、attention_mask、labels）
#   - desc: 进度条描述信息，便于追踪处理进度
train_dataset = dataset.map(dataset_process, desc="Processing...")

## 6. 模型加载  
加载预训练的 Deepseek-R1-1.5B 模型，设置为float16以节省显存。

In [ ]:
# 加载预训练的Causal Language Model（因果语言模型）
# 功能说明：
#   - 从指定路径加载预训练的因果语言模型（如Deepseek、Qwen等），用于下游微调或推理任务。
#   - 支持float16精度以节省显存，适合大模型训练和推理。
#   - 使用高效的注意力实现（sdpa）以提升推理速度和内存效率。
#
# 参数说明：
#   - model_path: 模型文件或目录的路径，需包含模型权重和配置文件。
#   - torch_dtype: 指定模型权重的数据类型（此处为torch.float16，节省显存）。
#   - attn_implementation: 注意力机制的实现方式（"sdpa"为高效稀疏注意力）。

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
).to(device)

在模型微调前，先用原始模型对同样的问题进行推理，观察其输出结果，为后续对比做准备。

In [ ]:
# 使用模型的 generate 方法进行文本生成推理
# 方法说明：
#   - model.generate 用于根据输入的 token 序列生成模型的输出文本，常用于推理和生成任务。
# 参数说明：
#   - input_ids: 输入文本的 token id 序列（通常为张量），作为生成的起始内容。
#   - attention_mask: 注意力掩码，指示哪些 token 需要被关注（1 表示有效，0 表示填充）。
#   - pad_token_id: 填充 token 的 id，用于对齐序列长度。
#   - max_new_tokens: 生成的最大新 token 数，控制输出文本的长度。
#   - do_sample: 是否采用采样方式生成（True 表示采样，适合生成多样化内容）。
#   - temperature: 采样温度，值越高生成内容越随机，越低则越确定。
#   - top_k: 采样时只考虑概率最高的 k 个 token，提升生成质量。
#   - top_p: 采样时只考虑累计概率达到 p 的 token 集合（nucleus sampling），进一步控制多样性。
generated_ids = model.generate(
    text_inputs.input_ids,
    attention_mask=text_inputs.attention_mask,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    top_k=10,
    top_p=0.95
)

将生成的内容转换为文本

In [ ]:
# 方法说明：
#   - tokenizer.batch_decode 用于将模型生成的 token id 序列批量解码为可读文本字符串。
# 参数说明：
#   - generated_ids: 需要解码的 token id 序列（通常为模型生成的输出）。
#   - skip_special_tokens: 是否跳过特殊token（如[CLS]、[SEP]等），True表示跳过，输出更干净的文本。
#   - clean_up_tokenization_spaces: 是否清理分词时产生的多余空格，True表示自动去除多余空格，使输出更自然。
decoded_texts = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

In [ ]:
decoded_texts

模型在 `<think>` 后输出了思考的过程，并最终输出了相应的答案。输出结果由于随机性会存在差异。

## 7. 模型微调

### 7.1 设定微调参数  

设置LoRA参数（秩r、alpha、目标模块、dropout等），并用 get_peft_model 将原始模型转换为LoRA微调模型。此时只有极少数参数可训练，大大降低资源消耗。

In [ ]:
# 方法说明：
# LoraConfig 用于配置LoRA（Low-Rank Adaptation）微调参数，帮助大模型在只训练极少量参数的情况下实现高效微调，显著降低显存和计算资源消耗。
# 主要参数说明：
#   - r: 秩（rank），低秩分解的秩，决定可训练参数的规模，数值越大表达能力越强，但资源消耗也增加。
#   - lora_alpha: LoRA缩放因子，用于调整LoRA层的输出幅度，影响训练稳定性和效果。
#   - target_modules: 需要注入LoRA结构的模块名称列表，通常为注意力层中的投影（如"q_proj"、"v_proj"）。
#     除了"q_proj"和"v_proj"，还可以根据模型结构添加如"k_proj"（键投影）、"o_proj"（输出投影）、"gate_proj"（门控投影）、"down_proj"（下投影）、"up_proj"（上投影）等模块。
#     具体可选模块需结合所用模型的实现细节（如Llama、ChatGLM、BERT等）。
#   - lora_dropout: LoRA层的dropout概率，有助于防止过拟合，提升泛化能力。
#   - bias: 是否对原始模型的bias参数进行微调，"none"表示不微调bias。
#   - task_type: 指定任务类型，这里为"CAUSAL_LM"（因果语言建模），适用于自回归生成任务。
peft_config = LoraConfig(
    r=32,  # LoRA秩，控制可训练参数量
    lora_alpha=64,  # LoRA缩放因子
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # 注入LoRA的目标模块
    lora_dropout=0.05,  # LoRA层的dropout概率
    bias="none",  # 不微调bias参数
    task_type="CAUSAL_LM"  # 任务类型为因果语言建模
)

In [ ]:
# 方法说明：
# get_peft_model 用于将原始大模型与LoRA配置（peft_config）结合，生成可微调的PEFT（Parameter-Efficient Fine-Tuning）模型。
# 这样只需训练极少量参数（如LoRA注入的低秩矩阵），大幅降低显存和计算资源消耗，适合大模型微调场景。
# 
# 参数说明：
#   - model: 原始的预训练大模型（如Llama、ChatGLM等）。
#   - peft_config: LoRA微调的配置参数（如秩r、alpha、目标模块、dropout等）。
model = get_peft_model(model, peft_config)

# print_trainable_parameters 方法用于打印当前模型中可训练参数的数量和比例，帮助我们确认LoRA微调后，只有极少部分参数参与训练，便于资源评估和调优。
model.print_trainable_parameters()

### 7.2 训练参数与回调设置  
配置训练参数（batch size、学习率、epoch数、混合精度等），并自定义回调类用于记录loss变化，便于后续可视化。

In [ ]:
# 方法说明：
# LossCallback 是一个自定义的训练回调类，继承自 TrainerCallback，用于在模型训练过程中自动记录每次 log 时的 loss 值，便于后续分析和可视化 loss 曲线，帮助判断模型收敛情况和训练效果。
# 
# 参数说明：
#   - self.losses: 用于保存每次 log 时记录下来的 loss 数值，类型为列表。
#   - self.epoch: 记录 log 被调用的次数，可用于统计训练过程中的步数或 epoch 数。
#   - on_log: 回调方法，每当 Trainer 触发 log 事件时自动调用。参数 logs 为当前日志信息（字典），如包含 "loss" 键，则将其值追加到 losses 列表中。
# 
# 使用方法：
# 实例化 LossCallback 后，将其作为回调传递给 Trainer，即可在训练过程中自动收集 loss 数据。

class LossCallback(TrainerCallback):
    """
    自定义训练回调类，用于记录训练过程中的 loss 变化。

    属性:
        losses (list): 存储每次 log 时的 loss 数值。
        epoch (int): 记录 log 被调用的次数。

    方法:
        on_log: 在每次 log 事件发生时，自动将 loss 记录到 losses 列表中。
    """
    def __init__(self):
        self.losses = []  # 用于保存 loss 数值
        self.epoch = 0    # 记录 log 调用次数

    def on_log(self, args, state, control, logs=None, **kwargs):
        """
        当 Trainer 触发 log 事件时自动调用，提取并保存 loss。

        参数:
            args, state, control: Trainer 传递的训练参数和状态信息。
            logs (dict): 当前日志信息，包含 loss 等指标。
            **kwargs: 其他可选参数。
        """
        if logs is not None and "loss" in logs:
            self.losses.append(logs["loss"])
        self.epoch += 1

# 实例化回调对象
loss_callback = LossCallback()

In [ ]:
# 方法说明：
# TrainingArguments 是 Huggingface Transformers 中用于配置 Trainer 训练过程的参数类。
# 通过设置 TrainingArguments，可以灵活控制模型训练的输出目录、批量大小、学习率、训练轮数、日志记录、优化器类型、混合精度等关键训练超参数。
# 合理配置这些参数有助于显存优化、训练效率提升以及训练过程的可追溯性。

# 参数说明：
#   - output_dir：训练输出文件保存路径。
#   - per_device_train_batch_size：每个设备（如每张显卡）上的训练 batch size。设置为1有助于降低单卡显存占用。
#   - gradient_accumulation_steps：梯度累计步数。实际等效于 batch_size = per_device_train_batch_size * gradient_accumulation_steps。
#   - num_train_epochs：训练总轮数（epoch）。
#   - learning_rate：优化器的初始学习率。
#   - fp16：是否启用混合精度训练（float16），可加速训练并节省显存。
#   - logging_steps：每隔多少步记录一次日志。
#   - save_strategy：模型保存策略。此处设置为 "no"，表示不自动保存模型。
#   - report_to：日志报告方式。设置为 "none" 表示不向外部系统（如wandb、tensorboard）报告。
#   - optim：优化器类型，这里采用 "adamw_torch"。
#   - dataloader_pin_memory：是否在 DataLoader 中使用 pin_memory，加速数据传输到GPU。
#   - remove_unused_columns：是否删除未被模型 forward 方法使用的数据列。

training_args = TrainingArguments(
    output_dir=output_path,
    per_device_train_batch_size=1,  # 显存优化设置
    gradient_accumulation_steps=4,  # 累计梯度，相当于总batch_size=4
    num_train_epochs=1,
    learning_rate=3e-4,
    fp16=True,  # 开启混合精度训练
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,
    remove_unused_columns=True  # 防止删除未使用的列
)

### 7.3 数据整理与Trainer初始化

定义数据整理函数，将处理好的数据打包成batch，初始化Trainer对象，传入模型、参数、数据集和回调。

In [ ]:
def data_collator(data):
    """
    数据整理函数（data_collator）

    作用：
        用于将单条样本数据打包成批量（batch），以便模型训练时高效地进行张量运算。
        该函数会将输入的每个样本的 input_ids、attention_mask 等字段转换为张量，并堆叠成批量张量，适配 Huggingface Trainer 的 batch 输入格式。

    参数说明：
        data: List[Dict]
            - 输入为一个样本字典的列表，每个字典包含 'input_ids'、'attention_mask' 等字段，通常由自定义 Dataset 返回。

    返回值：
        batch: Dict[str, torch.Tensor]
            - 返回一个字典，包含批量化的 'input_ids'、'attention_mask' 和 'labels'（此处 labels 直接等于 input_ids，适用于自回归语言建模任务）。

    注意事项：
        - 本函数假设每个样本的 input_ids 和 attention_mask 长度一致。
        - labels 字段用于计算损失，通常在自回归任务中与 input_ids 相同。
    """
    input_ids = torch.stack([torch.tensor(d["input_ids"]) for d in data])
    attention_mask = torch.stack([torch.tensor(d["attention_mask"]) for d in data])
    labels = torch.stack([torch.tensor(d["input_ids"]) for d in data]) # labels 与 input_ids 相同
    batch = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
    return batch


In [ ]:
# 下面代码用于初始化 Huggingface 的 Trainer 实例，用于模型的训练过程。
# Trainer 是 Huggingface Transformers 库中用于封装训练流程的高级接口，能够自动处理训练循环、梯度累积、日志记录、模型保存等功能。

# 参数说明：
# - model: 传入要训练的模型对象，Trainer 会自动调用其 forward 方法进行前向传播。
# - args: 训练参数配置（TrainingArguments 实例），包括学习率、batch size、训练轮数、输出目录等。
# - train_dataset: 训练用的数据集，需为 Dataset 或可迭代对象。
# - data_collator: 数据整理函数（如 data_collator），用于将单条样本打包成 batch，适配模型输入。
# - callbacks: 回调函数列表（如 [loss_callback]），可用于自定义训练过程中的行为（如记录损失、早停等）。

trainer = Trainer(
    model=model,                      # 需要训练的模型
    args=training_args,               # 训练参数配置
    train_dataset=train_dataset,     # 训练数据集
    data_collator=data_collator,      # 数据整理函数
    callbacks=[loss_callback]         # 回调函数列表
)

由于大模型微调需要 gpu，因此大家运行目录下的 `train.py`文件并使用 gpu 环境运行。

In [ ]:
trainer.train()
model.save_pretrained(output_path)  # 保存模型训练结果

### 7.5 测试微调效果  

微调完成后，加载LoRA权重，分别用微调前后的模型对同一问题进行推理，观察输出内容的变化。可以看到，微调后的模型在专业领域问题上的表现更贴合实际需求。

In [ ]:
# 本段代码用于加载已经训练好的LoRA（Low-Rank Adaptation）微调权重，并将其应用到基础大模型上，实现模型参数的高效微调与推理。
# 主要方法说明如下：

from peft import PeftModel, PeftConfig  

# PeftConfig.from_pretrained 用于从指定目录（output_path）加载LoRA微调的配置信息。
# 参数说明：
#   - output_path: 字符串，保存LoRA权重和配置的文件夹路径。
peft_config = PeftConfig.from_pretrained(output_path)

# PeftModel.from_pretrained 用于将LoRA权重加载到原始模型（model）上，得到融合了微调能力的新模型。
# 参数说明：
#   - model: 基础大模型对象，作为LoRA权重的载体。
#   - output_path: 字符串，LoRA权重文件的存放路径。
lora_model = PeftModel.from_pretrained(model, output_path)

使用相同的输入测试微调模型的效果

In [ ]:
generated_ids = lora_model.generate(
    text_inputs.input_ids,
    attention_mask=text_inputs.attention_mask,
    pad_token_id = tokenizer.pad_token_id,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    top_k=10,
    top_p=0.95
)

In [ ]:
decoded_texts = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

In [ ]:
decoded_texts

模型经过微调后输出更为专业的答案。  
大家可以通过下面方向重新训练效果更好的模型：  
1. 修改训练数据的指令格式，使其更契合模型的思考方式。
2. 调整 Lora 的配置参数，例如调整秩、alpha值等。
3. 调整训练参数，例如学习率、epoch 等。  

## 8. 总结

- LoRA微调能让大模型快速适应新任务，且资源消耗极低。
- 通过对比实验，微调后的模型在医学问答等专业领域表现更优。
- 掌握数据处理、模型训练、效果评估等完整流程，为后续大模型应用打下基础。